<a href="https://colab.research.google.com/github/VictorHugoTesti/FATEC-Processamento_Linguagem_Natural/blob/main/%5BPLN%5DAula08_ProjetoIntegrado0923.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalação e Carregamento de Bibliotecas

In [86]:
!pip install spacy
!python -m spacy download pt_core_news_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 11.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [87]:
import spacy
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity

In [88]:
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('rslp', quiet=True)

True

In [89]:
nlp = spacy.load("pt_core_news_md")

print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


# Pipeline

In [90]:
corpus = [
"Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.",
"Vocês trabalham com financiamento pela Caixa Econômica Federal para imóveis usados?",
"Gostaria de agendar uma visita ao imóvel do bairro Jardim Europa nesta semana.",
"Qual é o valor do condomínio e do IPTU do apartamento no centro de Araraquara?",
"A casa na Vila Mariana ainda está disponível para locação?",
"Aceitam fiador ou seguro fiança para o contrato de aluguel?",
"Sou a corretora Camila Souza, entrando em contato sobre o interesse do cliente João Pedro no imóvel de Ribeirão Preto.",
"O apartamento é novo ou já foi reformado recentemente?",
"Qual o prazo médio de aprovação do financiamento bancário?",
"Estou buscando um imóvel próximo ao Shopping Iguatemi, para compra à vista.",
"O sobrado no bairro Santa Cecília aceita animais de estimação?",
"Poderiam enviar mais fotos do apartamento e da planta baixa?",
"Qual é a taxa de juros aplicada pelo Banco do Brasil nesse tipo de financiamento?",
"A casa alugada tem vaga de garagem coberta?",
"Tenho interesse em investir em um imóvel para locação de longo prazo.",
"Poderia me indicar imóveis disponíveis na região central de Matão?",
]

for i, doc in enumerate(corpus, 1):
    print(f"Documento {i}: {doc}")

Documento 1: Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.
Documento 2: Vocês trabalham com financiamento pela Caixa Econômica Federal para imóveis usados?
Documento 3: Gostaria de agendar uma visita ao imóvel do bairro Jardim Europa nesta semana.
Documento 4: Qual é o valor do condomínio e do IPTU do apartamento no centro de Araraquara?
Documento 5: A casa na Vila Mariana ainda está disponível para locação?
Documento 6: Aceitam fiador ou seguro fiança para o contrato de aluguel?
Documento 7: Sou a corretora Camila Souza, entrando em contato sobre o interesse do cliente João Pedro no imóvel de Ribeirão Preto.
Documento 8: O apartamento é novo ou já foi reformado recentemente?
Documento 9: Qual o prazo médio de aprovação do financiamento bancário?
Documento 10: Estou buscando um imóvel próximo ao Shopping Iguatemi, para compra à vista.
Documento 11: O sobrado no bairro Santa Cecília aceita animais de estimação?
Doc

## Seção 1 - Tokenização

In [91]:
# Tokenizando todos os documentos do corpus com spaCy
docs_spacy = [nlp(doc) for doc in corpus]

In [92]:
print(" Tokens do Documento 0 ")
tokens_doc0 = [token.text for token in docs_spacy[0]]
print(tokens_doc0)

print("\n Tokens do Documento 3 ")
tokens_doc1 = [token.text for token in docs_spacy[3]]
print(tokens_doc1)

 Tokens do Documento 0 
['Bom', 'dia', '!', 'Meu', 'nome', 'é', 'Marcos', 'Ferreira', 'e', 'tenho', 'interesse', 'no', 'apartamento', 'anunciado', 'na', 'Rua', 'Voluntários', 'da', 'Pátria', ',', 'em', 'Matão', '.']

 Tokens do Documento 3 
['Qual', 'é', 'o', 'valor', 'do', 'condomínio', 'e', 'do', 'IPTU', 'do', 'apartamento', 'no', 'centro', 'de', 'Araraquara', '?']


Comentário: O tokenizador do spaCy separa corretamente os sinais de pontuação (como ! e ?) em tokens independentes, diferentemente de métodos simplistas como .split() que deixariam a pontuação grudada na palavra adjacente.

## Seção 2 - Stopwords, Stemming e Lematização

In [93]:
doc_escolhido = corpus[2]
print(f"Texto: {doc_escolhido}\n")

doc_nlp = nlp(doc_escolhido)

Texto: Gostaria de agendar uma visita ao imóvel do bairro Jardim Europa nesta semana.



In [94]:
# 1. Remoção de pontuação e stopwords
stop_pt = set(stopwords.words('portuguese'))
tokens_limpos = [t.text for t in doc_nlp if not t.is_punct and not t.is_stop]

In [95]:
# 2. Stemming (RSLP)
stemmer = RSLPStemmer()
stems = [stemmer.stem(t) for t in tokens_limpos]

In [96]:
# 3. Lematização (spaCy)
doc_tokens_limpos = nlp(" ".join(tokens_limpos))
lemas = [t.lemma_ for t in doc_tokens_limpos]

In [97]:
# Tabela Comparativa
df_comparacao = pd.DataFrame({
    "Token Limpo": tokens_limpos,
    "Stemming (RSLP)": stems,
    "Lematização (spaCy)": lemas
})

display(df_comparacao)

,Token Limpo,Stemming (RSLP),Lematização (spaCy)
0,Gostaria,gost,Gostaria
1,agendar,agend,agendar
2,visita,visit,visita
3,imóvel,imóvel,imóvel
4,bairro,bairr,bairro
5,Jardim,jardim,Jardim
6,Europa,europ,Europa
7,semana,seman,semanar


Comentário: É possível observar que o stemming aplica um corte mecânico de sufixos, gerando às vezes radicais truncados que não são palavras reais da língua portuguesa (ex.: agend ou imóvel transformados em radicais próprios do RSLP). Em contrapartida, a lematização utiliza dicionário e contexto gramatical para retornar a forma-raiz válida no dicionário.

## Seção 3 - Análise Sintática

In [98]:
indices_doc = [0, 4]

for idx in indices_doc:
    texto = corpus[idx]
    doc = nlp(texto)
    print(f"\n DOCUMENTO {idx}: ")
    print(f"Texto: {texto}\n")


 DOCUMENTO 0: 
Texto: Bom dia! Meu nome é Marcos Ferreira e tenho interesse no apartamento anunciado na Rua Voluntários da Pátria, em Matão.


 DOCUMENTO 4: 
Texto: A casa na Vila Mariana ainda está disponível para locação?



In [99]:
# Tabela de POS Tagging
dados_pos = [{"Token": t.text, "POS": t.pos_, "TAG": t.tag_, "DEP": t.dep_} for t in doc]
df_pos = pd.DataFrame(dados_pos)
display(df_pos)

,Token,POS,TAG,DEP
0,A,DET,DET,det
1,casa,NOUN,NOUN,nsubj
2,na,ADP,ADP,case
3,Vila,PROPN,PROPN,nmod
4,Mariana,PROPN,PROPN,flat:name
5,ainda,ADV,ADV,advmod
6,está,AUX,AUX,cop
7,disponível,ADJ,ADJ,ROOT
8,para,ADP,ADP,case
9,locação,NOUN,NOUN,obl


In [100]:
# Extração de SN e SV
raiz = [t for t in doc if t.dep_ == "ROOT"][0]
sujeito = [t for t in doc if t.dep_ == "nsubj"]
sujeito = sujeito[0] if sujeito else None

sintagma_nominal = list(sujeito.subtree) if sujeito else []
sintagma_verbal = [t for t in raiz.subtree if t not in sintagma_nominal]

print(f"Raiz (ROOT): {raiz.text} (POS: {raiz.pos_}, DEP: {raiz.dep_})")
print(f"Sintagma Nominal (SN): {[t.text for t in sintagma_nominal]}")
print(f"Sintagma Verbal (SV): {[t.text for t in sintagma_verbal]}")

Raiz (ROOT): disponível (POS: ADJ, DEP: ROOT)
Sintagma Nominal (SN): ['A', 'casa', 'na', 'Vila', 'Mariana']
Sintagma Verbal (SV): ['ainda', 'está', 'disponível', 'para', 'locação', '?']


Comentário: No Documento 0, a raiz é o verbo principal da oração ("tenho"). No Documento 4 ("A casa na Vila Mariana ainda está disponível para locação?"), o verbo principal é um verbo de ligação/copulativo ("está"), fazendo com que a árvore de dependência trate o predicativo do sujeito ("disponível", um adjetivo) como elemento central da estrutura sintática conforme os padrões do spaCy.

## Seção 4 - Interpretação Semântica

In [101]:
palavra1 = nlp("apartamento")
palavra2 = nlp("imóvel")
palavra3 = nlp("visita")
palavra4 = nlp("gato")

sim_1_2 = palavra1.similarity(palavra2)
sim_1_3 = palavra1.similarity(palavra3)
sim_1_4 = palavra1.similarity(palavra4)

print(f"Similaridade ('apartamento' x 'imóvel'): {sim_1_2:.4f}")
print(f"Similaridade ('apartamento' x 'visita'): {sim_1_3:.4f}")
print(f"Similaridade ('apartamento' x 'gato'): {sim_1_4:.4f}")

Similaridade ('apartamento' x 'imóvel'): 0.6667
Similaridade ('apartamento' x 'visita'): 0.2321
Similaridade ('apartamento' x 'gato'): 0.1860


Comentário: Os resultados fazem total sentido semântico: "apartamento" e "imóvel" apresentam alta similaridade por serem sinônimos diretos dentro do domínio imobiliário. Já "visita" apresenta uma proximidade moderada (ações comuns no mesmo ecossistema), enquanto "gato" aponta uma similaridade significativamente mais baixa por não possuir relação semântica com o nicho de mercado analisado.

## Seção 5 - Extração de Features

In [102]:
# Lematização do corpus inteiro
def lematizar_texto(texto):
    doc = nlp(texto)
    return " ".join([t.lemma_.lower() for t in doc if not t.is_punct and not t.is_space])

corpus_lematizado = [lematizar_texto(doc) for doc in corpus]
stop_pt = list(stopwords.words('portuguese'))

In [103]:
# Construção das matrizes Bag-of-Words e TF-IDF
vectorizer_bow = CountVectorizer(stop_words=stop_pt)
matriz_bow = vectorizer_bow.fit_transform(corpus_lematizado)

vectorizer_tfidf = TfidfVectorizer(stop_words=stop_pt)
matriz_tfidf = vectorizer_tfidf.fit_transform(corpus_lematizado)

In [104]:
# Análises
vocabulario_tfidf = vectorizer_tfidf.get_feature_names_out()
matriz_tfidf_array = matriz_tfidf.toarray()

In [105]:
# Documento com mais palavras únicas
soma_palavras_unicas_por_doc = (matriz_bow.toarray() > 0).sum(axis=1)
doc_mais_unico = np.argmax(soma_palavras_unicas_por_doc)

print(f"O documento com mais palavras únicas é o Documento {doc_mais_unico} com {soma_palavras_unicas_por_doc[doc_mais_unico]} termos distintos.")

O documento com mais palavras únicas é o Documento 0 com 13 termos distintos.


In [106]:
# Identificando maior peso TF-IDF
max_idx = np.unravel_index(np.argmax(matriz_tfidf_array), matriz_tfidf_array.shape)
palavra_maior_peso = vocabulario_tfidf[max_idx[1]]
doc_maior_peso = max_idx[0]
peso_encontrado = matriz_tfidf_array[max_idx]

print(f"A palavra com maior peso TF-IDF é '{palavra_maior_peso}' no Documento {doc_maior_peso} (Peso: {peso_encontrado:.4f}).")

A palavra com maior peso TF-IDF é 'novo' no Documento 7 (Peso: 0.5344).


Comentário: A lematização prévia reduziu drasticamente a esparsidade do vocabulário ao unificar flexões verbais e plurais. Termos específicos como nomes de cidades ("Matão", "Araraquara") ou instituições ("Caixa", "Iguatemi") destacam-se com maiores pesos TF-IDF por aparecerem com alta frequência em documentos isolados e baixa frequência no restante do corpus global.

## Seção 6 - KDT

In [107]:
# 1. Aplicando NER em 3 documentos contendo entidades (Docs: 0, 1 e 6)
docs_ner_indices = [0, 1, 6]

for idx in docs_ner_indices:
    doc = nlp(corpus[idx])
    print(f"Documento {idx} Entidades Nomeadas (NER):")
    if doc.ents:
        for ent in doc.ents:
            print(f"  - {ent.text} -> {ent.label_}")
    else:
        print("  - Nenhuma entidade detectada.")
    print("-" * 40)

Documento 0 Entidades Nomeadas (NER):
  - Marcos Ferreira -> PER
  - Rua Voluntários da Pátria -> LOC
  - Matão -> LOC
----------------------------------------
Documento 1 Entidades Nomeadas (NER):
  - Vocês -> LOC
  - Caixa Econômica Federal -> ORG
----------------------------------------
Documento 6 Entidades Nomeadas (NER):
  - Camila Souza -> PER
  - João Pedro -> PER
  - Ribeirão Preto -> LOC
----------------------------------------


In [108]:
# 2. Treinamento do modelo LDA
lda = LatentDirichletAllocation(n_components=2, random_state=42, max_iter=50)
lda.fit(matriz_bow)

LatentDirichletAllocation(max_iter=50, n_components=2, random_state=42)

In [109]:
# Exibindo as palavras mais representativas de cada tópico
vocabulario_bow = vectorizer_bow.get_feature_names_out()
print("\n Tópicos Descobertos pelo LDA: ")
for i, topico in enumerate(lda.components_):
    top_indices = topico.argsort()[-5:][::-1]
    palavras_topico = [vocabulario_bow[idx] for idx in top_indices]
    print(f"Tópico {i}: {palavras_topico}")


 Tópicos Descobertos pelo LDA: 
Tópico 0: ['imóvel', 'financiamento', 'apartamento', 'preto', 'sobre']
Tópico 1: ['imóvel', 'ter', 'apartamento', 'interesse', 'locação']


In [110]:
# Distribuindo tópicos por documento
distribuicao_topicos = lda.transform(matriz_bow)
print("\n Distribuição de Tópicos por Documento (Primeiros 3 docs): ")
for i in range(3):
    print(f"Documento {i} -> Tópico 0: {distribuicao_topicos[i][0]:.2f}, Tópico 1: {distribuicao_topicos[i][1]:.2f}")


 Distribuição de Tópicos por Documento (Primeiros 3 docs): 
Documento 0 -> Tópico 0: 0.04, Tópico 1: 0.96
Documento 1 -> Tópico 0: 0.08, Tópico 1: 0.92
Documento 2 -> Tópico 0: 0.94, Tópico 1: 0.06


Comentário: O NER conseguiu extrair com sucesso nomes de pessoas (PER), locais (LOC) e organizações (ORG), estruturando as menções brutas em unidades discretas de informação. O LDA agrupou coocorrências estatísticas do corpus, criando tópicos que refletem os temas latentes das mensagens dos leads (como transações financeiras/bancos versus características físicas e localização de imóveis).

## Seção 7 - Conclusão

As etapas iniciais de pré-processamento e estruturação linguística, como a tokenização via spaCy, a lematização para redução da esparsidade vocabular e a análise sintática de dependências, atuaram como pré-requisitos fundamentais e indispensáveis para a aplicação das técnicas estatísticas avançadas de extração de features e KDT. Sem a lematização prévia executada no corpus, por exemplo, variações flexionadas de palavras-chaves (como "imóvel" e "imóveis") gerariam colunas independentes e redundantes nas matrizes Bag-of-Words e TF-IDF, fragmentando o sinal estatístico necessário para que o algoritmo de aprendizado não supervisionado, conseguissem agrupar tópicos e padrões semânticos com precisão em textos do mundo real.